In [ ]:
# Importamos librerías
import base64
import urllib.request
import urllib.parse
import urllib.error
import json
from datetime import date
import numpy as np
from ast import literal_eval
import time

In [ ]:
def get_oauth_token():
    """ Obtiene el token de autenticación de Idealista usando urllib """
    api_key = 'xxxxxxxxxxxxxxxxxx'
    secret = 'xxxxxxxx'

    # Crear credenciales en formato Basic
    credentials = f"{api_key}:{secret}"
    credentials_b64 = base64.b64encode(credentials.encode()).decode()

    url = "https://api.idealista.com/oauth/token"

    # Preparar datos
    data = urllib.parse.urlencode({
        'grant_type': 'client_credentials',
        'scope': 'read'
    }).encode()

    headers = {
        'Authorization': f'Basic {credentials_b64}',
        'Content-Type': 'application/x-www-form-urlencoded'
    }

    req = urllib.request.Request(url, data=data, headers=headers)

    try:
        response = urllib.request.urlopen(req)
        token_data = json.loads(response.read().decode())
        print("Token obtenido correctamente")
        print(f"Expira en: {token_data['expires_in']} segundos")
        return token_data['access_token']
    except urllib.error.HTTPError as e:
        print(f"Error {e.code}")
        print(f"Response: {e.read().decode()}")
        return None

In [ ]:
# Obtener token
token = get_oauth_token()
print(f"Token: {token}...")  # Muestra solo los primeros 50 caracteres

In [ ]:
def obtener_todas_propiedades(token, filtros, max_paginas=None, pausa_segundos=1):
    """
    Obtiene propiedades de Idealista con paginación automática usando urllib
    """
    url = "https://api.idealista.com/3.5/es/search"

    headers = {
        'Authorization': f'Bearer {token}',
        'Content-Type': 'application/x-www-form-urlencoded'
    }

    todas_propiedades = []

    # Primera petición
    filtros_primera = filtros.copy()
    filtros_primera['numPage'] = 1
    filtros_primera['maxItems'] = 50

    print("Haciendo primera petición...")

    # Preparar datos
    data = urllib.parse.urlencode(filtros_primera).encode()
    req = urllib.request.Request(url, data=data, headers=headers)

    try:
        response = urllib.request.urlopen(req)
        datos = json.loads(response.read().decode())

        total_propiedades = datos.get('total', 0)
        total_paginas = datos.get('totalPages', 0)

        print(f"Total de propiedades encontradas: {total_propiedades}")
        print(f"Total de páginas: {total_paginas}")

        # Añadir primera página
        todas_propiedades.extend(datos.get('elementList', []))
        print(f"Página 1/{total_paginas}: {len(datos.get('elementList', []))} propiedades")

        # Limitar páginas si es necesario
        if max_paginas:
            total_paginas = min(total_paginas, max_paginas)
            print(f"Limitando descarga a {max_paginas} páginas")

        # Descargar resto de páginas
        for num_pagina in range(2, total_paginas + 1):
            filtros_pagina = filtros.copy()
            filtros_pagina['numPage'] = num_pagina
            filtros_pagina['maxItems'] = 50

            # Pausa para no saturar la API
            time.sleep(pausa_segundos)

            data = urllib.parse.urlencode(filtros_pagina).encode()
            req = urllib.request.Request(url, data=data, headers=headers)

            try:
                response = urllib.request.urlopen(req)
                datos = json.loads(response.read().decode())
                propiedades = datos.get('elementList', [])
                todas_propiedades.extend(propiedades)

                print(f"Página {num_pagina}/{total_paginas}: {len(propiedades)} propiedades. Total acumulado: {len(todas_propiedades)}")

            except urllib.error.HTTPError as e:
                if e.code == 429:
                    print("Límite alcanzado. Esperando 60 segundos...")
                    time.sleep(60)
                    continue
                else:
                    print(f"Error en página {num_pagina}: {e.code}")
                    break

        print(f"\nDescarga completada: {len(todas_propiedades)} propiedades en total")
        return todas_propiedades

    except urllib.error.HTTPError as e:
        print(f"Error: {e.code} - {e.read().decode()}")
        return []

In [ ]:
# Define tus filtros de búsqueda
filtros = {
    'country': 'es',
    'operation': 'sale',    # 'sale' o 'rent'
    'propertyType': 'homes',   # 'homes', 'offices', 'premises', 'garages', 'bedrooms'
    'center': '39.8581,-4.02263',  # Coordenadas (latitud,longitud) - Toledo centro
    'distance': '4000',          # Radio en metros
    # Filtros opcionales:
    # 'minPrice': 200000,
    # 'maxPrice': 500000,
    # 'bedrooms': '2,3',        # 2 o 3 habitaciones
    # 'elevator': True,
    # 'minSize': 80,
    # 'maxSize': 120,
}

print("Filtros configurados:")
for clave, valor in filtros.items():
    print(f"  - {clave}: {valor}")

In [ ]:
# Buscar propiedades
# max_paginas=None descarga TODAS (cuidado con el límite de 100 requests/mes)
# max_paginas=5 descarga solo 5 páginas (250 propiedades máximo)

propiedades = obtener_todas_propiedades(
    token=token,
    filtros=filtros,
    max_paginas=None,      # Cambia a un número para limitar (ej: 5)
    pausa_segundos=1       # Pausa entre peticiones
)

print(f"\nPropiedades descargadas: {len(propiedades)}")

In [ ]:
# Ver la primera propiedad como ejemplo
if propiedades:
    print("Ejemplo de propiedad:")
    print(json.dumps(propiedades[0], indent=2, ensure_ascii=False))

In [ ]:
import pandas as pd

# Convertir a DataFrame para análisis
df = pd.DataFrame(propiedades)

print(f"DataFrame creado con {len(df)} propiedades")
print(f"\nColumnas disponibles:")
print(df.columns.tolist())

# Ver primeras filas
df.head()

In [ ]:
# Guardar en JSON
with open('propiedades_idealista_alquiler_habitacion.json', 'w', encoding='utf-8') as f:
    json.dump(propiedades, f, ensure_ascii=False, indent=2)
print("Guardado en propiedades_idealista_habitacion.json")

# Guardar en CSV
df.to_csv('propiedades_idealista_venta_casa.csv', index=False, encoding='utf-8')
print("Guardado en propiedades_idealista.csv")

# Guardar en Excel
df.to_excel('propiedades_idealista_habitacion.xlsx', index=False)
print("Guardado en propiedades_idealista.xlsx")

In [ ]:
# Calcular cuántas requests has hecho
num_paginas = len(propiedades) / 50 if propiedades else 0
num_paginas = int(num_paginas) + (1 if len(propiedades) % 50 > 0 else 0)

print(f"Requests realizados: {num_paginas}")
print(f"Requests restantes (aprox): {100 - num_paginas}")